# 🗣️ Dialogue Summarizer — Fine-tuning FLAN-T5 on DialogSum

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jamir-hossain/dialogue-summarizer/blob/main/notebooks/Dialogue_Summarizer_FineTune.ipynb)

This notebook is the **training half** of the project:

1. Fine-tune [`google/flan-t5-base`](https://huggingface.co/google/flan-t5-base) on [DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum) (12k+ real-life dialogues with human-written summaries).
2. Measure quality with **ROUGE** before and after fine-tuning.
3. Export the model to **ONNX** (+ int8 quantization) so it runs in **Node.js** with Transformers.js.
4. Upload it to the **Hugging Face Hub** — the Node.js app in `app/` loads it from there.

> **Runtime:** `Runtime → Change runtime type → T4 GPU`. With the default settings (4,000 dialogues, 2 epochs) training takes roughly 15–30 minutes on a free T4.

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q "transformers==4.57.6" "evaluate==0.4.6" "optimum[onnx]" onnxruntime rouge_score sentencepiece accelerate datasets nltk

> If Colab asks you to **restart the session** after installing, do it, then continue from the next cell.

## 2. Configuration
Change the model or data size here. Use `google/flan-t5-small` for a quick run, or `TRAIN_SAMPLES = 12460` for the full dataset.

In [ ]:
import os

# ---------------- Project settings (change here) ----------------
MODEL_CHECKPOINT = os.environ.get("MODEL_CHECKPOINT", "google/flan-t5-base")   # pre-trained model from the Hugging Face Hub
DATASET_NAME = "knkarthick/dialogsum"
TRAIN_SAMPLES = int(os.environ.get("TRAIN_SAMPLES", 4000))   # full train set = 12,460 dialogues
EVAL_SAMPLES = int(os.environ.get("EVAL_SAMPLES", 200))      # ROUGE is computed on generated text, which is slow
NUM_EPOCHS = float(os.environ.get("NUM_EPOCHS", 2))
PREFIX = "summarize: "                                        # T5 models expect the task name as a prefix
MAX_INPUT_TOKENS = 512
MAX_TARGET_TOKENS = 128
OUTPUT_DIR = "dialogue-summarizer-model"
print(MODEL_CHECKPOINT, TRAIN_SAMPLES, EVAL_SAMPLES, NUM_EPOCHS)

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 3. Dataset — DialogSum
Each example has a `dialogue` (input) and a human-written `summary` (target).

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME)
print(dataset)

sample = dataset["test"][0]
print("\nDIALOGUE:\n", sample["dialogue"])
print("\nHUMAN SUMMARY:\n", sample["summary"])

In [ ]:
train_ds = dataset["train"].shuffle(seed=42).select(range(TRAIN_SAMPLES))
eval_ds = dataset["validation"].shuffle(seed=42).select(range(EVAL_SAMPLES))
print("Train:", len(train_ds), "| Eval:", len(eval_ds))

## 4. Load the pre-trained model and try it *before* fine-tuning

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT).to(device)
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
def summarize(dialogue, max_new_tokens=MAX_TARGET_TOKENS):
    inputs = tokenizer(PREFIX + dialogue, return_tensors="pt",
                       max_length=MAX_INPUT_TOKENS, truncation=True).to(device)
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("BEFORE fine-tuning:\n", summarize(sample["dialogue"]))

## 5. Tokenize
Dialogues become `input_ids`/`attention_mask`; summaries become `labels`.

In [ ]:
def preprocess(batch):
    model_inputs = tokenizer([PREFIX + d for d in batch["dialogue"]],
                             max_length=MAX_INPUT_TOKENS, truncation=True)
    labels = tokenizer(text_target=batch["summary"], max_length=MAX_TARGET_TOKENS, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

columns = dataset["train"].column_names
tokenized_train = train_ds.map(preprocess, batched=True, remove_columns=columns)
tokenized_eval = eval_ds.map(preprocess, batched=True, remove_columns=columns)
print(tokenized_train)
print("labels of first example:", tokenized_train[0]["labels"][:15], "...")

## 6. ROUGE metric
ROUGE measures word overlap between the model's summary and the human summary (0–100, higher is better).

In [ ]:
import numpy as np
import evaluate
import nltk

nltk.download("punkt_tab", quiet=True)
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # rougeLsum expects one sentence per line
    decoded_preds = ["\n".join(nltk.sent_tokenize(p.strip())) for p in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(l.strip())) for l in decoded_labels]
    scores = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 2) for k, v in scores.items()}

## 7. Trainer

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=25,
    predict_with_generate=True,          # generate real summaries during evaluation (needed for ROUGE)
    generation_max_length=MAX_TARGET_TOKENS,
    report_to="none",
    seed=42,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

### Baseline ROUGE (before fine-tuning)

In [ ]:
before = trainer.evaluate()
print({k: v for k, v in before.items() if k.startswith("eval_rouge")})

## 8. Fine-tune 🚀

In [ ]:
train_result = trainer.train()
print("Training loss:", round(train_result.training_loss, 3),
      "| minutes:", round(train_result.metrics["train_runtime"] / 60, 1))

### ROUGE after fine-tuning

In [ ]:
after = trainer.evaluate()
print({k: v for k, v in after.items() if k.startswith("eval_rouge")})

In [ ]:
import matplotlib.pyplot as plt

names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
b = [before[f"eval_{n}"] for n in names]
a = [after[f"eval_{n}"] for n in names]
logs = [l for l in trainer.state.log_history if "loss" in l]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(names))
r1 = ax1.bar(x - 0.2, b, 0.4, label="Before fine-tuning", color="#95a5a6")
r2 = ax1.bar(x + 0.2, a, 0.4, label="After fine-tuning", color="#27ae60")
ax1.bar_label(r1, fmt="%.1f"); ax1.bar_label(r2, fmt="%.1f")
ax1.set_xticks(x, names); ax1.set_ylabel("ROUGE (0-100)")
ax1.set_title("ROUGE on validation dialogues"); ax1.legend()
ax2.plot([l["step"] for l in logs], [l["loss"] for l in logs], marker="o", color="#c0392b")
ax2.set_xlabel("Training step"); ax2.set_ylabel("Training loss"); ax2.set_title("Training loss")
plt.tight_layout()
plt.savefig("results.png", dpi=120)
plt.show()

## 9. Try it on test dialogues and on your own conversation

In [ ]:
# DialogSum's test split stores every dialogue 3 times (3 human summaries), so take every 3rd row
for i in range(0, 9, 3):
    ex = dataset["test"][i]
    print(f"--- Test dialogue {i // 3} ---")
    print("HUMAN :", ex["summary"])
    print("MODEL :", summarize(ex["dialogue"]))
    print()

my_dialogue = """#Person1#: Hi Rafi, are you coming to the team meeting at 3 pm?
#Person2#: I can't, my bus broke down near Farmgate. I'll be at least an hour late.
#Person1#: No problem. Should I record the meeting for you?
#Person2#: Yes please, and send me the slides too.
#Person1#: Sure, I'll email everything after the meeting."""
print("MY DIALOGUE ->", summarize(my_dialogue))

## 10. Save the fine-tuned model

In [ ]:
trainer.save_model(OUTPUT_DIR)          # saves model weights, config and tokenizer
print(sorted(os.listdir(OUTPUT_DIR)))

## 11. Export to ONNX for Node.js (Transformers.js)
Transformers.js expects `config.json` + tokenizer files at the top level and the ONNX files inside an `onnx/` folder.

In [ ]:
import subprocess, shutil

ONNX_RAW = "onnx_raw"
subprocess.run(["optimum-cli", "export", "onnx", "--model", OUTPUT_DIR,
                "--task", "text2text-generation-with-past", ONNX_RAW], check=True)
print(sorted(os.listdir(ONNX_RAW)))

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

JS_MODEL_DIR = "dialogue-summarizer-onnx"           # this folder is what the Node.js app loads
os.makedirs(f"{JS_MODEL_DIR}/onnx", exist_ok=True)

for f in ["config.json", "generation_config.json", "tokenizer.json",
          "tokenizer_config.json", "special_tokens_map.json", "spiece.model"]:
    shutil.copy(f"{ONNX_RAW}/{f}", JS_MODEL_DIR)

for name in ["encoder_model", "decoder_model_merged"]:
    shutil.copy(f"{ONNX_RAW}/{name}.onnx", f"{JS_MODEL_DIR}/onnx/{name}.onnx")          # fp32
    quantize_dynamic(f"{ONNX_RAW}/{name}.onnx", f"{JS_MODEL_DIR}/onnx/{name}_quantized.onnx",
                     weight_type=QuantType.QUInt8,
                     extra_options={"EnableSubgraph": True})                               # int8 weights: ~4x smaller, faster on CPU

for root, _, files in os.walk(JS_MODEL_DIR):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"{p:<60} {os.path.getsize(p) / 1e6:8.1f} MB")

## 12. Upload to the Hugging Face Hub
You need a Hugging Face **access token with write permission** (huggingface.co → Settings → Access Tokens).
In Colab you can store it as a secret named `HF_TOKEN` (🔑 icon on the left) — `login()` will ask for it otherwise.

In [ ]:
from huggingface_hub import login, HfApi

login()
api = HfApi()
repo_id = api.create_repo("dialogue-summarizer-onnx", exist_ok=True).repo_id

card = f"""---
library_name: transformers.js
pipeline_tag: summarization
base_model: {MODEL_CHECKPOINT}
datasets:
- {DATASET_NAME}
language:
- en
license: apache-2.0
---

# Dialogue Summarizer (ONNX)

`{MODEL_CHECKPOINT}` fine-tuned on DialogSum ({TRAIN_SAMPLES} dialogues, {NUM_EPOCHS:g} epochs) and exported to ONNX for Transformers.js.

| Metric | Before fine-tuning | After fine-tuning |
|---|---|---|
""" + "\n".join(f"| {n} | {before['eval_' + n]:.1f} | {after['eval_' + n]:.1f} |" for n in names) + """

Input format: prefix the dialogue with `summarize: `.
"""
open(f"{JS_MODEL_DIR}/README.md", "w").write(card)

api.upload_folder(folder_path=JS_MODEL_DIR, repo_id=repo_id)
print("Model page:", f"https://huggingface.co/{repo_id}")

Alternatively, download the exported folder and put it in `app/models/` of the Node.js app:

In [ ]:
import shutil
from google.colab import files

shutil.make_archive(JS_MODEL_DIR, "zip", JS_MODEL_DIR)
files.download(f"{JS_MODEL_DIR}.zip")

## 13. (Optional) Run the Node.js app right here in Colab
Colab machines have Node.js installed. The app needs **Node 20+** — the first cell upgrades it if necessary.
The repository URL below points to the project on GitHub.

In [ ]:
!node -v
!if [ "$(node -v | cut -c2- | cut -d. -f1)" -lt 20 ]; then curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null && apt-get install -y nodejs >/dev/null; fi
!node -v

In [ ]:
!git clone -q https://github.com/jamir-hossain/dialogue-summarizer.git
!cp -r dialogue-summarizer-onnx dialogue-summarizer/app/models/
!cd dialogue-summarizer/app && npm install --silent && npm test

In [ ]:
%%bash
cd dialogue-summarizer/app
node src/cli.js <<'EOF'
#Person1#: Can you pick up Ayesha from school today? I have a late meeting.
#Person2#: Sure, what time does her class end?
#Person1#: At 4:30. And please buy some bread on the way back.
#Person2#: Okay, I'll text you when we're home.
EOF